# MIRA: Multimodal Infrastructure Risk Analyzer
This project is a POC for ACM Research;
Lead: Advay Chandramouli


## Import Requisite Libraries

In [ ]:
import pandas as pd
df = pd.read_csv("data/im3_open_source_data_center_atlas.csv")
df.shape
df.head()

## Exploratory Data Analysis (EDA)

In [ ]:
unique_operators = sorted(
    df["operator"].fillna("").replace("", "(Not specified)").unique()
)
print(f"{len(unique_operators)} unique operators:")
pd.DataFrame(unique_operators, columns=["operator"])

In [ ]:
import plotly.express as px

# Records by Operator (top 20 for readability; long names work best on horizontal bars)
operator_counts = (
    df["operator"]
    .fillna("")
    .replace("", "(Not specified)")
    .value_counts()
    .reset_index()
)
operator_counts.columns = ["operator", "count"]
operator_counts["pct"] = (
    operator_counts["count"] / operator_counts["count"].sum() * 100
).round(1)
operator_top = operator_counts.head(20).sort_values("count")
operator_top["label"] = operator_top.apply(
    lambda row: f"{row['count']} ({row['pct']}%)", axis=1
)

fig_ops = px.bar(
    operator_top,
    x="count",
    y="operator",
    orientation="h",
    title="Records by Operator (Top 20)",
    labels={"count": "Number of Data Centers", "operator": "Operator"},
    text="label",
    color="count",
    color_continuous_scale="Blues",
)
fig_ops.update_layout(
    yaxis={"categoryorder": "total ascending"},
    showlegend=False,
    coloraxis_showscale=False,
    height=640,
    margin={"l": 20, "r": 40, "t": 60, "b": 40},
    plot_bgcolor="white",
)
fig_ops.update_traces(textposition="outside", cliponaxis=False)
fig_ops.show()

# Records by State (horizontal bar chart with counts and percentages)
state_counts = df["state"].value_counts().reset_index()
state_counts.columns = ["state", "count"]
state_counts["pct"] = (state_counts["count"] / state_counts["count"].sum() * 100).round(
    1
)
state_sorted = state_counts.sort_values("count")
state_sorted["label"] = state_sorted.apply(
    lambda row: f"{row['count']} ({row['pct']}%)", axis=1
)

fig_states = px.bar(
    state_sorted,
    x="count",
    y="state",
    orientation="h",
    title="Records by State",
    labels={"count": "Number of Data Centers", "state": "State"},
    text="label",
    color="count",
    color_continuous_scale="Teal",
)
fig_states.update_layout(
    yaxis={"categoryorder": "total ascending"},
    showlegend=False,
    coloraxis_showscale=False,
    height=1100,
    margin={"l": 20, "r": 40, "t": 60, "b": 40},
    plot_bgcolor="white",
)
fig_states.update_traces(textposition="outside", cliponaxis=False)
fig_states.show()

## Tabular Preprocessing

In [ ]:
df = df[df["type"] == "building"]
df_buildings = df.drop(columns=["state_abb", "state_id", "county", "county_id", "ref"])
df_buildings["sqft"] = df_buildings["sqft"].astype(int)

In [ ]:
print(f"Shape: {df_buildings.shape}")
print(f"Columns: {list(df_buildings.columns)}")
df_buildings.head()

## WRI Aqueduct Input Format

In [ ]:
example_coordinates = pd.read_csv("data/example_coordinates.csv")
print("Example coordinates format:")
example_coordinates

wri_aqueduct_df = df_buildings[["id", "name", "lat", "lon"]].copy()
wri_aqueduct_df.columns = example_coordinates.columns

print(f"Columns: {list(wri_aqueduct_df.columns)}")
print(f"Shape: {wri_aqueduct_df.shape}")
wri_aqueduct_df.head()

In [ ]:
sample_df = wri_aqueduct_df.head(5).copy()
sample_df.to_csv("data/sample_df.csv", index=False)
sample_df

In [ ]:
wri_top5_data = pd.read_csv("data/wri_top5_data.csv")
print(f"Columns ({len(wri_top5_data.columns)}):")
print(list(wri_top5_data.columns))

print("\nColumn types:")
wri_top5_data.dtypes